# 🏙️ Chicago 311: Urban Issue Classification 

Welcome to our Machine Learning Study Jam! Today, we are stepping into the shoes of Data Scientists and AI Engineers for the City of Chicago. 

## 🎯 The Problem Statement
The city receives thousands of 311 service requests every month—ranging from broken streetlights to abandoned vehicles. Currently, dispatchers have to manually read and categorize these reports. 

**Our Goal:** We are going to build a **Predictive Triage AI**. We want to teach a Machine Learning model to automatically classify the **Type of Service Request** based entirely on *where* it happened (Coordinates) and *when* it happened (Time of year/week).

## 📌 Details to Take Note Of
Before we write any code, as engineers, we must understand our constraints:
* **The Target:** We are predicting the `sr_type` (Service Request Type). To keep it simple, we will only train the AI to recognize the Top 3 most common issues (e.g., Potholes, Street Lights Out, Graffiti).
* **The Features:** We are restricting the AI's "vision" to just four data points: `Latitude`, `Longitude`, `Month`, and `Day of the Week`. 
* **The Challenge:** City data is messy. Potholes are highly seasonal (happening more in certain months), and some neighborhoods report issues more frequently than others. Our model needs to find these hidden patterns.

## 🗺️ The Study Jam Roadmap
We will build this project in four distinct phases:

### **Phase 1: Data Ingestion & Exploration**
* Connect to the Microsoft Azure Open Datasets cloud.
* Load a historical slice of Chicago Safety data.
* Clean the data by dropping rows with missing GPS coordinates.

### **Phase 2: Feature Engineering**
* AI models understand numbers, not text or calendar dates. 
* We will extract the "Month" and "Day of the Week" from the raw timestamp to give our model a sense of time.
* We will convert our text labels (like "Pothole") into numerical categories.

### **Phase 3: Model Training (The AI Brain)**
* We will split our dataset into a **Training Set** (80% to teach the AI) and a **Testing Set** (20% to quiz the AI).
* We will initialize and train a **Random Forest Classifier**—an algorithm excellent at finding geographic and decision-based boundaries.

### **Phase 4: Evaluation & Insights**
* We will test our AI against the 20% of data it has never seen.
* We will generate a **Classification Report** and a **Confusion Matrix** to visualize exactly where our model is getting confused (e.g., is it mistaking graffiti for abandoned vehicles?).


In [ ]:
'''
Starter from https://learn.microsoft.com/en-us/azure/open-datasets/dataset-chicago-safety?tabs=azureml-opendatasets
'''

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This package is in preview.
from azureml.opendatasets import ChicagoSafety
from datetime import datetime
from dateutil import parser

end_date = parser.parse('2016-01-01')
start_date = parser.parse('2015-05-01')
safety = ChicagoSafety(start_date=start_date, end_date=end_date)
safety = safety.to_pandas_dataframe()

In [ ]:
# Explore the datset accordingly.
dataset = safety
print(f" How many rows are the dataset: {len(dataset)}")
print(f" How many columns are the dataset: {len(dataset.columns)}")
print(f" Rows and columns: {dataset.shape}")
print(f" Dataset columns:\n {dataset.columns}")

In [ ]:
# To visually view the contents of the dataset, you may use:
print(f"First 5 columns:")
dataset.head()

In [ ]:
# But there are times where you'll want to treat your columns as rows to save up some space.
# In that case, just use this: 
print(f"First 5 columns transposed:\n ")
dataset.head().transpose()

In [ ]:
# Btw, the numbers in the right are indices from Pandas. It's basically index numbers. Don't worry, the models just ignore that.
# However, if it bothers you visually, you can reset it to look clean (0, 1, 2, 3...) using:
dataset.reset_index(drop=True).head()

In [ ]:
# Now, let's check the details of our dataset. 
# We have a total of 56909 entries and in the same time we can observe how many entries are..
# non-null or non-missing values and their datatypes.
# float64 are just float numeric numbers, datetime is the formatted date, and object is basically apart from those -- could be strings.
dataset.info()

In [ ]:
print(f"How many unique values for each column:\n{dataset.nunique()}")

# In my observation here datatype, extended properties, and subtype holds only 1 value. 
# So they aren't that relevant because we can't map patterns with only 1 value.
# We will just drop them later on.

In [ ]:
# Let's move forward now to our target column - category.
dataset['category'].value_counts()

In [ ]:
# I noticed category seems off, given they are really complants or service report.
# let's check them first then rename it, then check.
dataset.rename(columns={'category': 'complaint_type'}).head()


In [ ]:
# But take note, why then if we look at our original dataset, category column wasn't changed?
dataset.head(3)

In [ ]:
# Simply because we didn't put the command in place. 
# There are two ways in doing this. You may either do 
# dataset = dataset.rename(columns={'category': 'complaint_type'})
# or...
dataset.rename(columns={'category': 'complaint_type'}, inplace=True)
dataset.head()
# Please take note, however, that dataset = dataset.rename(columns={'category': 'complaint_type'}) is the standard practice.
# So moving forward, avoid using inplace.

# Now our dataset is now changed

In [ ]:
# Count values
complaint_counts = dataset['complaint_type'].value_counts()

# Plot
plt.figure(figsize=(8, 5))
complaint_counts.plot(kind='bar')

# Labels and title
plt.title('Distribution of Complaint Types')
plt.xlabel('Complaint Type')
plt.ylabel('Count')

# Rotate labels if needed
plt.xticks(rotation=0)

plt.show()

## Data Cleaning

In [ ]:
# Let's first create a copy just in case we have to edit the original dataset later
dataset_clean = dataset.copy()

In [ ]:
# Going back earlier, we identified datatype, subtype, subcategory, source, extended properties as either null columns or irrelevant to our use-case.
# Therefore, we will drop them:
dataset_clean = dataset_clean.drop(columns=['dataType', 'dataSubtype',  'subcategory', 'extendedProperties', 'source'])
dataset_clean.head()

In [ ]:
# Now, let's check for null or missing values. 
dataset_clean.isnull().sum()

# The machine can't use null values in its training. Therefore, we either drop them or impute them
# Impute is basically filling up those null or missing values by either median, mean, or zero depending on the data
# Since latitude and longtitude is geographic in nature, we wll not impute them and just drop them.


In [ ]:
dataset_clean = dataset_clean.dropna(subset=['latitude', 'longitude'])
dataset_clean.head(3)

In [ ]:
# let's check again if it was removed accoridngly.
dataset_clean.isnull().sum()

# And yes, it was dropped.

In [ ]:
# We might as well check for duplicates
print(f"Nmber of duplicates: {dataset_clean.duplicated().sum()}")
# There are several reasons as to why there are duplicates in our data.
# Given this context, I think it might be because several people have reported the same problem or the system recorded the incident multiple times
# This is particularly bad because the model may think that the incident recorded 10 times may signify as 10x more important or likely.
# We might not like that in our use-case, right?

In [ ]:
# So what we will do? Let's just drop them.
dataset_clean = dataset_clean.drop_duplicates()
print(f"Nmber of duplicates: {dataset_clean.duplicated().sum()}")

# Now, we will move forward to feature engineering.

## Feature Engineering

In [ ]:
# So feature engineering is basically engineering the columns so the model can be trained better. 
# This includes creating new columns out of existing values or generally, reorganizing your code to extract insights.
# The most typical example of a column where we can do this is those columns composed of a date.
# So now, let's extract the month and day...and maybe the name of the day.

dataset_clean['month'] = dataset_clean['dateTime'].dt.month
dataset_clean['day_of_week'] = dataset_clean['dateTime'].dt.day_of_week
dataset_clean['name_of_day'] = dataset_clean['dateTime'].dt.day_name()
dataset_clean.head(3)

In [ ]:
# I actually don't think that getting identifying whether it's weekened or not will be a big factor
# But for the sake of demonstration, let's do it.
# Please take note, that it's still a behavioural signal
# For instance, people may report more often in weekends than in weekdays.
# Let's check it afterwards

dataset_clean['is_weekend'] = np.where(dataset_clean['name_of_day'].isin(['Saturday', 'Sunday']), 1, 0)
# Basically, that says if it's in col 'name_of_day' and the entry is in 'saturday' or 'sunday', 
# then we will put 1. Otherwise, 0
dataset_clean.head()



In [ ]:
# Let's visualize it as well
weekly_distribution = dataset_clean['is_weekend'].value_counts()

# Plot
plt.figure(figsize=(8, 5))
weekly_distribution.plot(kind='bar')

# Labels and title
plt.title('Distribution of Complaints (Weekend v Weekdays)')
plt.xlabel('Weekly Composition')
plt.ylabel('Count')

# Rotate labels if needed
plt.xticks(rotation=0)

plt.show()

In [ ]:
dataset_clean = dataset_clean.drop(columns=['dateTime', 'status', 'address', 'name_of_day', 'day_of_week'])
dataset_clean.head()

In [ ]:
# For the sake of convention, you know what, let's rename the complaint_type as the target since that's what we are predicting
dataset_clean = dataset_clean.rename(columns={'complaint_type': 'target'})

## Model Training

In [ ]:
# Let's now split our dataset from the target and the input features

X = dataset_clean.drop(columns='target')
Y = dataset_clean['target']

# Basically, X is all columns that's not the target
# And Y is just the column target. So it will just be series

In [ ]:
#  We will split our data: 80% goes to the Training Set (The Study Guide), and 20% goes to the Testing Set (The Final Exam).
from sklearn.model_selection import train_test_split

# This is just by convention so don't worry much.
# Train-Test-Split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state=42)

In [ ]:
# As you know, our dataset have categorical columns or non-integer values
# Obviously, the model can't read non-int numbers so what we will do here is make those values represented by integers
# Thanks to sklearn, we can do that
from sklearn.preprocessing import LabelEncoder

label_enc = LabelEncoder()
Y_train = label_enc.fit_transform(Y_train)
Y_test = label_enc.transform(Y_test)

In [ ]:
# So okay, what just we've done. Basically, we transofrmed the complaint into 1, 2, 3
# Wanna see it?

id_to_label = dict(enumerate(label_enc.classes_))
print(id_to_label)

# So basically, the model will output 1, then we will translate that, for instance as Pothole in Street 

In [ ]:
from sklearn.ensemble import RandomForestClassifier

RFC = RandomForestClassifier(
    random_state=42
)


